In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, matthews_corrcoef, roc_curve, roc_auc_score,
    average_precision_score
)
from sklearn.model_selection import train_test_split

import os
os.makedirs("Metrics/XGB", exist_ok=True)
os.makedirs("Saved Models/XGB", exist_ok=True)

from Feature_Extraction import compute_aac, compute_dpc, compute_tpc, compute_pcp, one_hot_encode

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit          # replaces train_test_split

df = pd.read_csv("balanced_named.csv")

X = df.drop(columns=["label", "sequence", "cluster"])          # keep the two new columns out of the features
y = df["label"]
groups = df["cluster"]                                         # +

feature_names = X.columns      # used later by the SHAP section

# random split: cluster membership is deliberately ignored
train_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=0.2, stratify=y, random_state=42)

# each cluster goes entirely to train or entirely to test
#splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
#train_idx, test_idx = next(splitter.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("features:", X.shape[1], "(should be 9036)")
print("train:", len(y_train), " test:", len(y_test))
print("test label balance:\n", y_test.value_counts())

features: 9036 (should be 9036)
train: 5292  test: 1324
test label balance:
 label
1    662
0    662
Name: count, dtype: int64


In [17]:
# === Pipeline ===
pipe_xgb = ImbPipeline([
    ("select", SelectKBest(score_func=f_classif, k=20)),
    ("scale",  StandardScaler()),
    ("clf",    XGBClassifier(
        learning_rate=0.14017769458977059,
        max_depth=7,
        n_estimators=357,
        subsample=0.8609993861334124,
        colsample_bytree=0.5714334089609704,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    ))
])

# === Fit & Predict ===
pipe_xgb.fit(X_train, y_train)
y_pred = pipe_xgb.predict(X_test)
y_prob = pipe_xgb.predict_proba(X_test)[:, 1]

# ============
# Print report
# ============
tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()

print("=== XGBoost Stand-Alone Model ===")
print(f"Accuracy:    {accuracy_score(y_test, y_pred):.4f}")
print(f"MCC:         {matthews_corrcoef(y_test, y_pred):.4f}")
print(f"ROC-AUC:     {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC:      {average_precision_score(y_test, y_prob):.4f}")
print(f"Sensitivity: {tp/(tp+fn):.4f}")
print(f"Specificity: {tn/(tn+fp):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=4))

# === Confusion Matrix ===
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("XGBoost - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig("Metrics\\XGB\\xgb_confusion_matrix.png")
plt.close()

# === ROC Curve ===
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_prob):.4f}")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("XGBoost - ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig("Metrics\\XGB\\xgb_roc_curve.png")
plt.close()

# === SHAP Analysis ===
selector = pipe_xgb.named_steps["select"]
scaler = pipe_xgb.named_steps["scale"]
clf = pipe_xgb.named_steps["clf"]

X_sample = X_test[:100]
X_selected = selector.transform(X_sample)
X_scaled = scaler.transform(X_selected)

# Feature names for plot
selected_feature_names = X_train.columns[selector.get_support(indices=True)]

# Use TreeExplainer for XGBoost
explainer = shap.KernelExplainer(clf.predict, X_scaled)
shap_values = explainer(X_scaled)

# SHAP Summary Plot
plt.figure()
shap.summary_plot(
    shap_values, features=X_scaled,
    feature_names=selected_feature_names,
    show=False
)
plt.tight_layout()
plt.savefig("Metrics\\XGB\\xgb_shap_summary.png")
plt.close()

# === Save Model ===
joblib.dump(pipe_xgb, "Saved Models\\XGB\\xgb_pipeline_model.pkl")
print("Model saved to: Saved Models/XGB/xgb_pipeline_model.pkl")


c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 639  678  718  786  798  870  958  978 1018 1026 1030 1037 1038 1039
 1070 1090 1098 1192 1193 1218 1318 1378 1426 1430 1458 1510 1513 1558
 1581 1586 1701 1718 1758 1838 1938 1980 1981 1986 1998 2050 2058 2098
 2158 2178 2226 2227 2230 2239 2258 2290 2310 2370 2378 2383 2390 2396
 2397 2398 2399 2406 2838 2850 2918 2958 2970 2991 3020 3030 3035 3036
 3038 3050 3070 3098 3118 3150 3159 3178 3180 3186 3189 3194 3195 3196
 3197 3198 3199 3350 3430 3438 3441 3484 3593 3824 3826 4158 4178 4224
 4298 4381 4398 4446 4450 4459 4464 4472 4473 4478 4498 4512 4518 4538
 4541 4546 4550 4551 4552 4553 4554 4555 4558 4559 4590 4598 4621 4626
 4627 4637 4638 4644 4670 4678 4685 4698 4706 4718 4730 4758 4766 4778
 4782 4784 4789 4790 4792 4793 4796 4798 4799 4804 4806 4810 4811 4817
 4818 5038 5070 5110 5183 5186 5198 5250 5258 5298 5358 5438 5610 5618
 5659 5706 5758 5821 5831 5

=== XGBoost Stand-Alone Model ===
Accuracy:    0.9486
MCC:         0.8977
ROC-AUC:     0.9841
PR-AUC:      0.9873
Sensitivity: 0.9335
Specificity: 0.9637

Classification Report:
               precision    recall  f1-score   support

           0     0.9355    0.9637    0.9494       662
           1     0.9626    0.9335    0.9479       662

    accuracy                         0.9486      1324
   macro avg     0.9491    0.9486    0.9486      1324
weighted avg     0.9491    0.9486    0.9486      1324



  0%|          | 0/100 [00:00<?, ?it/s]

Model saved to: Saved Models/XGB/xgb_pipeline_model.pkl


In [26]:
import os
import numpy as np
import shap
import matplotlib.pyplot as plt

os.makedirs("Metrics/XGB", exist_ok=True)

# the same transforms the pipeline applies before the classifier
X_test_selected_scaled = pipe_xgb.named_steps["scale"].transform(
    pipe_xgb.named_steps["select"].transform(X_test)
)
feature_names = X.columns[pipe_xgb.named_steps["select"].get_support()]

explainer = shap.TreeExplainer(pipe_xgb.named_steps["clf"])
shap_values = explainer.shap_values(X_test_selected_scaled)

# newer SHAP returns (n_samples, n_features, n_classes) for binary models
sv = shap_values[:, :, 1] if np.ndim(shap_values) == 3 else shap_values

plt.figure()
shap.summary_plot(sv, X_test_selected_scaled,
                  feature_names=feature_names, show=False)
plt.tight_layout()
plt.savefig("Metrics/XGB/shap_xgb.png", dpi=200)
plt.close()

sel = X.columns[pipe_xgb.named_steps["select"].get_support()]
print(len(sel))
print(pd.Series([c.split("_")[0] for c in sel]).value_counts())
print(sorted(sel.tolist()))

20
DPC    7
PCP    7
AAC    5
BE     1
Name: count, dtype: int64
['AAC_C', 'AAC_D', 'AAC_E', 'AAC_G', 'AAC_K', 'BE_pos1_G', 'DPC_CK', 'DPC_KA', 'DPC_KC', 'DPC_KG', 'DPC_KI', 'DPC_KK', 'DPC_KL', 'PCP_Beta_strand_indices', 'PCP_Buriability', 'PCP_Hydrophilicity_scale', 'PCP_Hydrophobicity_index', 'PCP_Localized_electrical_effect', 'PCP_Net_charge', 'PCP_Relative_mutability']


In [18]:
# ==========================================
# Bootstrap Confidence Intervals for Metrics
# ==========================================

M = {"Accuracy": lambda t, p, s: accuracy_score(t, p),
     "MCC":      lambda t, p, s: matthews_corrcoef(t, p),
     "ROC-AUC":  lambda t, p, s: roc_auc_score(t, s),
     "PR-AUC":   lambda t, p, s: average_precision_score(t, s)}

yt = np.asarray(y_test)
rng = np.random.default_rng(42)
boot = {k: [] for k in M}

for _ in range(2000):
    i = rng.integers(0, len(yt), len(yt))
    if len(np.unique(yt[i])) < 2:
        continue
    for k, f in M.items():
        boot[k].append(f(yt[i], y_pred[i], y_prob[i]))

for k, f in M.items():
    lo, hi = np.percentile(boot[k], [2.5, 97.5])
    print(f"{k:9s} {f(yt, y_pred, y_prob):.4f}  95% CI [{lo:.4f}, {hi:.4f}]")

Accuracy  0.9486  95% CI [0.9366, 0.9600]
MCC       0.8977  95% CI [0.8739, 0.9200]
ROC-AUC   0.9841  95% CI [0.9774, 0.9899]
PR-AUC    0.9873  95% CI [0.9822, 0.9916]


In [27]:
# ========================================
# External Validation on Benchmark Dataset
# ========================================

import pandas as pd
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             matthews_corrcoef, roc_auc_score, average_precision_score)

external = pd.read_csv("external_benchmark.csv")

ext_X = pd.DataFrame(
    [compute_aac(s) + compute_dpc(s) + compute_tpc(s) + compute_pcp(s)
     + list(one_hot_encode(s)) for s in external["sequence"]],
    columns=X.columns, dtype="float32")
ext_y = external["label"].to_numpy()

ext_prob = pipe_xgb.predict_proba(ext_X)[:, 1]
ext_pred = (ext_prob >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(ext_y, ext_pred, labels=[0, 1]).ravel()

print("=== XGBoost — External Benchmark ===")
print(f"Accuracy:    {accuracy_score(ext_y, ext_pred):.4f}")
print(f"MCC:         {matthews_corrcoef(ext_y, ext_pred):.4f}")
print(f"ROC-AUC:     {roc_auc_score(ext_y, ext_prob):.4f}")
print(f"PR-AUC:      {average_precision_score(ext_y, ext_prob):.4f}")
print(f"Sensitivity: {tp/(tp+fn):.4f}")
print(f"Specificity: {tn/(tn+fp):.4f}")
print("\nClassification Report:\n", classification_report(ext_y, ext_pred, digits=4))

=== XGBoost — External Benchmark ===
Accuracy:    0.7696
MCC:         0.5527
ROC-AUC:     0.8449
PR-AUC:      0.8319
Sensitivity: 0.8792
Specificity: 0.6600

Classification Report:
               precision    recall  f1-score   support

           0     0.8453    0.6600    0.7413       803
           1     0.7211    0.8792    0.7924       803

    accuracy                         0.7696      1606
   macro avg     0.7832    0.7696    0.7668      1606
weighted avg     0.7832    0.7696    0.7668      1606



In [19]:
# ================
# Feature Ablation
# ================

from sklearn.base import clone

for drop in [None, "AAC", "DPC", "TPC", "PCP", "BE"]:
    cols = [c for c in X.columns if c.split("_")[0] != drop]
    Xa = X[cols]
    p = clone(pipe_xgb)                      # change to your pipeline name
    p.fit(Xa.iloc[train_idx], y.iloc[train_idx])
    pred = p.predict(Xa.iloc[test_idx])
    name = "ALL" if drop is None else f"without {drop}"
    print(f"{name:14s} {len(cols):5d} cols  "
          f"acc={accuracy_score(y.iloc[test_idx], pred):.4f}  "
          f"MCC={matthews_corrcoef(y.iloc[test_idx], pred):.4f}")

c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 639  678  718  786  798  870  958  978 1018 1026 1030 1037 1038 1039
 1070 1090 1098 1192 1193 1218 1318 1378 1426 1430 1458 1510 1513 1558
 1581 1586 1701 1718 1758 1838 1938 1980 1981 1986 1998 2050 2058 2098
 2158 2178 2226 2227 2230 2239 2258 2290 2310 2370 2378 2383 2390 2396
 2397 2398 2399 2406 2838 2850 2918 2958 2970 2991 3020 3030 3035 3036
 3038 3050 3070 3098 3118 3150 3159 3178 3180 3186 3189 3194 3195 3196
 3197 3198 3199 3350 3430 3438 3441 3484 3593 3824 3826 4158 4178 4224
 4298 4381 4398 4446 4450 4459 4464 4472 4473 4478 4498 4512 4518 4538
 4541 4546 4550 4551 4552 4553 4554 4555 4558 4559 4590 4598 4621 4626
 4627 4637 4638 4644 4670 4678 4685 4698 4706 4718 4730 4758 4766 4778
 4782 4784 4789 4790 4792 4793 4796 4798 4799 4804 4806 4810 4811 4817
 4818 5038 5070 5110 5183 5186 5198 5250 5258 5298 5358 5438 5610 5618
 5659 5706 5758 5821 5831 5

ALL             9036 cols  acc=0.9486  MCC=0.8977


c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 619  658  698  766  778  850  938  958  998 1006 1010 1017 1018 1019
 1050 1070 1078 1172 1173 1198 1298 1358 1406 1410 1438 1490 1493 1538
 1561 1566 1681 1698 1738 1818 1918 1960 1961 1966 1978 2030 2038 2078
 2138 2158 2206 2207 2210 2219 2238 2270 2290 2350 2358 2363 2370 2376
 2377 2378 2379 2386 2818 2830 2898 2938 2950 2971 3000 3010 3015 3016
 3018 3030 3050 3078 3098 3130 3139 3158 3160 3166 3169 3174 3175 3176
 3177 3178 3179 3330 3410 3418 3421 3464 3573 3804 3806 4138 4158 4204
 4278 4361 4378 4426 4430 4439 4444 4452 4453 4458 4478 4492 4498 4518
 4521 4526 4530 4531 4532 4533 4534 4535 4538 4539 4570 4578 4601 4606
 4607 4617 4618 4624 4650 4658 4665 4678 4686 4698 4710 4738 4746 4758
 4762 4764 4769 4770 4772 4773 4776 4778 4779 4784 4786 4790 4791 4797
 4798 5018 5050 5090 5163 5166 5178 5230 5238 5278 5338 5418 5590 5598
 5639 5686 5738 5801 5811 5

without AAC     9016 cols  acc=0.9350  MCC=0.8703


c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 239  278  318  386  398  470  558  578  618  626  630  637  638  639
  670  690  698  792  793  818  918  978 1026 1030 1058 1110 1113 1158
 1181 1186 1301 1318 1358 1438 1538 1580 1581 1586 1598 1650 1658 1698
 1758 1778 1826 1827 1830 1839 1858 1890 1910 1970 1978 1983 1990 1996
 1997 1998 1999 2006 2438 2450 2518 2558 2570 2591 2620 2630 2635 2636
 2638 2650 2670 2698 2718 2750 2759 2778 2780 2786 2789 2794 2795 2796
 2797 2798 2799 2950 3030 3038 3041 3084 3193 3424 3426 3758 3778 3824
 3898 3981 3998 4046 4050 4059 4064 4072 4073 4078 4098 4112 4118 4138
 4141 4146 4150 4151 4152 4153 4154 4155 4158 4159 4190 4198 4221 4226
 4227 4237 4238 4244 4270 4278 4285 4298 4306 4318 4330 4358 4366 4378
 4382 4384 4389 4390 4392 4393 4396 4398 4399 4404 4406 4410 4411 4417
 4418 4638 4670 4710 4783 4786 4798 4850 4858 4898 4958 5038 5210 5218
 5259 5306 5358 5421 5431 5

without DPC     8636 cols  acc=0.9403  MCC=0.8809


... (2 outputs left out)

c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 639  678  718  786  798  870  958  978 1018 1026 1030 1037 1038 1039
 1070 1090 1098 1192 1193 1218 1318 1378 1426 1430 1458 1510 1513 1558
 1581 1586 1701 1718 1758 1838 1938 1980 1981 1986 1998 2050 2058 2098
 2158 2178 2226 2227 2230 2239 2258 2290 2310 2370 2378 2383 2390 2396
 2397 2398 2399 2406 2838 2850 2918 2958 2970 2991 3020 3030 3035 3036
 3038 3050 3070 3098 3118 3150 3159 3178 3180 3186 3189 3194 3195 3196
 3197 3198 3199 3350 3430 3438 3441 3484 3593 3824 3826 4158 4178 4224
 4298 4381 4398 4446 4450 4459 4464 4472 4473 4478 4498 4512 4518 4538
 4541 4546 4550 4551 4552 4553 4554 4555 4558 4559 4590 4598 4621 4626
 4627 4637 4638 4644 4670 4678 4685 4698 4706 4718 4730 4758 4766 4778
 4782 4784 4789 4790 4792 4793 4796 4798 4799 4804 4806 4810 4811 4817
 4818 5038 5070 5110 5183 5186 5198 5250 5258 5298 5358 5438 5610 5618
 5659 5706 5758 5821 5831 5

without PCP     9020 cols  acc=0.9313  MCC=0.8628


c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 639  678  718  786  798  870  958  978 1018 1026 1030 1037 1038 1039
 1070 1090 1098 1192 1193 1218 1318 1378 1426 1430 1458 1510 1513 1558
 1581 1586 1701 1718 1758 1838 1938 1980 1981 1986 1998 2050 2058 2098
 2158 2178 2226 2227 2230 2239 2258 2290 2310 2370 2378 2383 2390 2396
 2397 2398 2399 2406 2838 2850 2918 2958 2970 2991 3020 3030 3035 3036
 3038 3050 3070 3098 3118 3150 3159 3178 3180 3186 3189 3194 3195 3196
 3197 3198 3199 3350 3430 3438 3441 3484 3593 3824 3826 4158 4178 4224
 4298 4381 4398 4446 4450 4459 4464 4472 4473 4478 4498 4512 4518 4538
 4541 4546 4550 4551 4552 4553 4554 4555 4558 4559 4590 4598 4621 4626
 4627 4637 4638 4644 4670 4678 4685 4698 4706 4718 4730 4758 4766 4778
 4782 4784 4789 4790 4792 4793 4796 4798 4799 4804 4806 4810 4811 4817
 4818 5038 5070 5110 5183 5186 5198 5250 5258 5298 5358 5438 5610 5618
 5659 5706 5758 5821 5831 5

without BE      8436 cols  acc=0.9471  MCC=0.8947


In [6]:
from sklearn.feature_selection import SelectKBest

selector = next(step for _, step in pipe_xgb.steps if isinstance(step, SelectKBest))
chosen_features = X.columns[selector.get_support()]

print("number of selected features:", len(chosen_features))
print(pd.Series([c.split("_")[0] for c in chosen_features]).value_counts())
print(list(chosen_features))

number of selected features: 20
DPC    8
PCP    6
AAC    5
BE     1
Name: count, dtype: int64
['AAC_C', 'AAC_D', 'AAC_E', 'AAC_G', 'AAC_K', 'DPC_CK', 'DPC_KA', 'DPC_KC', 'DPC_KG', 'DPC_KI', 'DPC_KK', 'DPC_KL', 'DPC_RG', 'PCP_Hydrophobicity_index', 'PCP_Hydrophilicity_scale', 'PCP_Buriability', 'PCP_Beta_strand_indices', 'PCP_Relative_mutability', 'PCP_Net_charge', 'BE_pos1_G']


In [20]:
# ================
# Repeated Seeds
# ================

accs, mccs = [], []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                  .split(X, y, groups))
    p = clone(pipe_xgb)                      # change to your pipeline name
    p.fit(X.iloc[tr], y.iloc[tr])
    pred = p.predict(X.iloc[te])
    accs.append(accuracy_score(y.iloc[te], pred))
    mccs.append(matthews_corrcoef(y.iloc[te], pred))
    print(f"seed {seed}: acc={accs[-1]:.4f}  MCC={mccs[-1]:.4f}")

print(f"\nacc {np.mean(accs):.4f} ± {np.std(accs, ddof=1):.4f}")
print(f"MCC {np.mean(mccs):.4f} ± {np.std(mccs, ddof=1):.4f}")

c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 621  638  782  786  798  853  870  906  958  978 1018 1024 1026 1030
 1037 1038 1039 1098 1189 1192 1193 1218 1348 1426 1458 1510 1511 1513
 1558 1581 1586 1701 1718 1758 1826 1980 1981 1986 1992 1998 2018 2050
 2051 2070 2141 2158 2178 2226 2227 2230 2258 2290 2310 2378 2383 2387
 2390 2398 2399 2413 2838 2850 2918 2942 2951 2963 2970 2991 3020 3030
 3035 3036 3038 3050 3070 3098 3150 3153 3178 3180 3185 3186 3194 3195
 3196 3197 3198 3199 3216 3258 3298 3350 3438 3441 3510 3592 3824 3826
 4158 4178 4298 4381 4398 4446 4450 4458 4459 4464 4472 4473 4498 4512
 4518 4538 4541 4550 4551 4552 4553 4554 4558 4559 4570 4590 4598 4621
 4627 4629 4636 4638 4639 4650 4666 4678 4698 4706 4718 4719 4730 4739
 4758 4778 4782 4784 4789 4790 4792 4793 4796 4798 4799 4801 4804 4809
 4810 4811 4815 4818 4850 5038 5070 5110 5186 5198 5250 5258 5298 5346
 5350 5356 5358 5430 5438 5

seed 0: acc=0.9297  MCC=0.8589


c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 718  786  798  870  878  958  978 1018 1022 1026 1030 1037 1038 1039
 1098 1106 1170 1186 1192 1193 1213 1218 1318 1351 1426 1458 1510 1511
 1513 1558 1581 1586 1590 1701 1718 1758 1904 1980 1981 1986 1998 2050
 2158 2178 2226 2227 2230 2258 2290 2310 2378 2383 2387 2389 2390 2397
 2398 2399 2838 2850 2870 2878 2918 2970 2991 3020 3030 3035 3036 3038
 3050 3059 3070 3098 3136 3150 3156 3178 3180 3186 3194 3195 3196 3197
 3198 3199 3210 3298 3350 3430 3438 3441 3824 3826 3986 3996 4006 4158
 4178 4298 4381 4398 4446 4447 4450 4459 4464 4472 4473 4478 4498 4512
 4518 4538 4541 4550 4551 4552 4553 4554 4558 4559 4590 4598 4621 4627
 4638 4644 4650 4658 4678 4685 4698 4706 4718 4719 4730 4758 4773 4778
 4782 4784 4790 4791 4792 4793 4796 4798 4799 4804 4807 4810 4811 4818
 4819 4950 5038 5070 5110 5186 5198 5218 5250 5298 5358 5421 5426 5438
 5570 5610 5618 5659 5758 5

seed 1: acc=0.9486  MCC=0.8970


c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 639  718  786  798  870  958  978 1018 1022 1024 1026 1030 1037 1038
 1039 1090 1098 1106 1192 1193 1194 1213 1218 1426 1430 1458 1510 1513
 1558 1581 1586 1701 1718 1758 1838 1858 1980 1981 1986 1990 1998 2050
 2158 2178 2226 2227 2230 2239 2250 2258 2290 2310 2378 2383 2390 2396
 2398 2399 2838 2850 2858 2918 2970 2991 3020 3026 3030 3032 3035 3036
 3038 3050 3070 3084 3092 3094 3098 3150 3178 3180 3186 3187 3194 3195
 3196 3197 3198 3199 3350 3438 3441 3510 3706 3824 3826 4158 4178 4298
 4381 4398 4446 4447 4450 4459 4464 4472 4473 4478 4498 4504 4510 4512
 4518 4538 4541 4546 4550 4551 4552 4553 4554 4558 4559 4590 4598 4621
 4626 4627 4638 4639 4640 4666 4678 4679 4685 4698 4706 4718 4719 4726
 4730 4746 4750 4758 4761 4778 4780 4782 4783 4784 4790 4792 4793 4796
 4798 4799 4804 4806 4810 4811 4818 5026 5032 5038 5070 5110 5186 5198
 5246 5250 5298 5346 5350 5

seed 2: acc=0.9441  MCC=0.8878


... (10 outputs left out)

c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 782  786  798  858  870  878  958  978 1018 1026 1030 1037 1038 1039
 1098 1106 1184 1192 1193 1218 1318 1426 1446 1458 1510 1513 1558 1581
 1586 1701 1718 1758 1838 1839 1858 1980 1981 1986 1990 1992 1998 2050
 2058 2070 2090 2141 2158 2178 2226 2227 2229 2230 2258 2290 2310 2378
 2383 2390 2397 2398 2399 2838 2850 2918 2958 2970 2991 3020 3030 3035
 3036 3038 3050 3070 3098 3118 3133 3146 3150 3159 3178 3180 3186 3187
 3194 3195 3196 3197 3198 3199 3218 3253 3258 3298 3350 3438 3441 3590
 3599 3601 3824 3826 3834 4158 4178 4298 4381 4387 4398 4446 4450 4459
 4464 4472 4473 4498 4506 4512 4518 4538 4541 4542 4544 4545 4550 4551
 4552 4553 4554 4556 4558 4559 4590 4598 4606 4621 4624 4627 4637 4638
 4678 4679 4684 4698 4706 4718 4730 4758 4761 4778 4782 4784 4789 4790
 4792 4793 4796 4798 4799 4804 4810 4811 4816 4818 5038 5070 5110 5186
 5198 5250 5298 5343 5346 5

seed 8: acc=0.9455  MCC=0.8912


c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 632  638  718  786  798  870  878  958  978 1018 1024 1026 1030 1037
 1038 1039 1046 1084 1098 1192 1193 1212 1218 1426 1458 1510 1513 1558
 1581 1586 1701 1718 1758 1826 1858 1938 1980 1981 1982 1986 1998 2050
 2110 2141 2144 2153 2158 2178 2224 2226 2227 2229 2230 2234 2239 2258
 2290 2310 2378 2383 2390 2398 2399 2406 2443 2558 2786 2832 2838 2850
 2893 2918 2970 2991 3020 3023 3030 3035 3036 3038 3050 3070 3090 3098
 3099 3107 3150 3158 3178 3180 3185 3186 3189 3194 3195 3196 3197 3198
 3199 3210 3244 3350 3438 3441 3459 3822 3824 3826 3834 3996 4018 4158
 4178 4224 4238 4298 4381 4398 4399 4446 4450 4459 4464 4472 4473 4478
 4498 4508 4512 4513 4518 4538 4540 4541 4548 4550 4551 4552 4553 4554
 4555 4558 4559 4570 4590 4598 4599 4621 4627 4638 4678 4698 4706 4718
 4730 4758 4778 4782 4784 4790 4792 4793 4796 4798 4799 4804 4806 4807
 4810 4811 4818 4901 4950 5

seed 9: acc=0.9503  MCC=0.9005

acc 0.9381 ± 0.0100
MCC 0.8760 ± 0.0201


In [21]:
import numpy as np
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, matthews_corrcoef

accs, mccs = [], []
for seed in range(10):
    tr, te = train_test_split(np.arange(len(df)), test_size=0.2,
                              stratify=y, random_state=seed)
    p = clone(pipe_xgb)
    p.fit(X.iloc[tr], y.iloc[tr])
    pred = p.predict(X.iloc[te])
    accs.append(accuracy_score(y.iloc[te], pred))
    mccs.append(matthews_corrcoef(y.iloc[te], pred))
    print(f"seed {seed}: acc={accs[-1]:.4f}  MCC={mccs[-1]:.4f}")

print(f"\nacc {np.mean(accs):.4f} ± {np.std(accs, ddof=1):.4f}")
print(f"MCC {np.mean(mccs):.4f} ± {np.std(mccs, ddof=1):.4f}")

c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 544  786  798  863  870  958  978 1018 1026 1030 1037 1038 1039 1098
 1106 1150 1186 1192 1193 1218 1426 1439 1458 1510 1513 1558 1581 1586
 1701 1718 1758 1980 1981 1986 1998 2018 2050 2158 2178 2224 2226 2227
 2230 2258 2290 2310 2370 2378 2383 2390 2398 2399 2478 2838 2850 2901
 2918 2938 2970 2991 3020 3030 3035 3036 3038 3050 3070 3098 3150 3159
 3178 3180 3185 3186 3194 3195 3196 3197 3198 3199 3210 3253 3350 3424
 3438 3441 3584 3824 3826 3838 3986 4158 4178 4224 4298 4381 4387 4398
 4446 4450 4459 4464 4472 4473 4498 4512 4518 4538 4541 4548 4550 4551
 4552 4553 4554 4558 4559 4590 4598 4621 4624 4627 4636 4638 4639 4678
 4698 4706 4718 4730 4758 4778 4782 4784 4786 4790 4792 4793 4796 4798
 4799 4804 4808 4810 4811 4818 4856 4950 5026 5038 5070 5110 5186 5198
 5218 5250 5298 5358 5430 5438 5610 5618 5659 5744 5758 5822 5826 5839
 5970 5983 6226 6229 6236 6

seed 0: acc=0.9486  MCC=0.8974


c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 786  798  870  898  926  958  977  978 1018 1026 1030 1037 1038 1039
 1090 1098 1192 1193 1218 1426 1458 1461 1510 1513 1558 1581 1586 1701
 1718 1758 1838 1980 1981 1982 1983 1986 1998 2050 2058 2158 2178 2224
 2226 2227 2230 2246 2258 2290 2310 2378 2383 2384 2387 2389 2390 2396
 2398 2399 2698 2786 2838 2850 2918 2938 2942 2958 2970 2991 3020 3023
 3024 3030 3035 3036 3038 3050 3070 3098 3118 3150 3178 3180 3185 3186
 3194 3195 3196 3197 3198 3199 3216 3258 3298 3350 3424 3438 3441 3490
 3586 3590 3824 3826 4158 4178 4218 4298 4381 4398 4446 4450 4459 4464
 4472 4473 4492 4498 4510 4512 4518 4538 4541 4546 4550 4551 4552 4553
 4554 4558 4559 4590 4598 4621 4624 4626 4627 4638 4658 4670 4678 4685
 4698 4706 4718 4719 4730 4739 4747 4758 4778 4782 4783 4784 4789 4790
 4791 4792 4793 4796 4798 4799 4804 4807 4810 4811 4818 4850 5032 5038
 5070 5110 5184 5186 5198 5

seed 1: acc=0.9471  MCC=0.8947


c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 638  678  786  798  870  940  958  978 1018 1024 1026 1030 1037 1038
 1039 1098 1192 1193 1218 1426 1439 1458 1510 1511 1513 1558 1581 1586
 1701 1718 1758 1838 1839 1980 1981 1986 1996 1998 2018 2050 2098 2158
 2178 2224 2226 2227 2230 2236 2258 2290 2310 2370 2378 2383 2390 2398
 2399 2826 2838 2850 2901 2918 2958 2970 2991 3020 3030 3035 3036 3038
 3050 3070 3098 3150 3178 3180 3186 3189 3190 3194 3195 3196 3197 3198
 3199 3310 3350 3438 3441 3484 3824 3826 3838 4158 4178 4224 4226 4298
 4381 4392 4398 4446 4450 4459 4464 4472 4473 4498 4504 4512 4518 4538
 4541 4546 4550 4551 4552 4553 4554 4555 4556 4558 4559 4590 4598 4618
 4621 4624 4626 4627 4638 4670 4678 4698 4706 4718 4719 4726 4730 4758
 4778 4782 4783 4784 4786 4790 4792 4793 4796 4798 4799 4804 4808 4810
 4811 4818 5026 5038 5070 5110 5186 5198 5250 5258 5298 5358 5438 5490
 5500 5610 5618 5659 5719 5

seed 2: acc=0.9539  MCC=0.9086


... (10 outputs left out)

c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 544  786  798  870  958  978 1018 1026 1030 1033 1037 1038 1039 1070
 1098 1192 1193 1210 1218 1348 1426 1458 1510 1513 1558 1581 1582 1586
 1701 1718 1758 1839 1980 1981 1986 1998 2050 2158 2178 2226 2227 2230
 2258 2290 2310 2378 2383 2389 2390 2398 2399 2698 2838 2850 2918 2951
 2970 2991 3020 3030 3032 3035 3036 3038 3050 3064 3066 3070 3098 3150
 3178 3180 3186 3187 3194 3195 3196 3197 3198 3199 3216 3258 3350 3424
 3430 3438 3441 3581 3592 3599 3824 3826 3986 4018 4158 4178 4224 4238
 4298 4359 4381 4398 4446 4450 4459 4464 4472 4473 4498 4508 4512 4518
 4538 4539 4541 4550 4551 4552 4553 4554 4558 4559 4570 4590 4598 4621
 4627 4638 4650 4658 4678 4698 4706 4718 4730 4758 4778 4782 4784 4787
 4790 4792 4793 4796 4798 4799 4804 4810 4811 4818 5032 5038 5070 5110
 5186 5198 5250 5258 5298 5358 5359 5428 5438 5610 5618 5659 5718 5758
 5766 5835 5970 5982 5983 5

seed 8: acc=0.9532  MCC=0.9068


c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 678  786  798  870  958  978 1018 1024 1026 1030 1033 1037 1038 1039
 1098 1106 1192 1193 1210 1213 1218 1426 1430 1458 1510 1513 1558 1581
 1582 1586 1594 1701 1718 1758 1798 1938 1980 1981 1986 1998 2050 2118
 2119 2153 2158 2178 2226 2227 2230 2236 2258 2290 2310 2378 2383 2389
 2390 2398 2399 2418 2838 2850 2918 2938 2970 2991 3020 3030 3035 3036
 3038 3050 3070 3098 3107 3150 3153 3178 3180 3186 3187 3190 3194 3195
 3196 3197 3198 3199 3216 3258 3350 3378 3438 3441 3484 3590 3599 3650
 3751 3824 3826 4018 4158 4178 4298 4381 4387 4398 4446 4447 4450 4459
 4464 4472 4473 4498 4512 4513 4518 4538 4541 4550 4551 4552 4553 4554
 4558 4559 4590 4598 4604 4621 4627 4638 4658 4678 4679 4698 4706 4718
 4730 4758 4778 4782 4784 4789 4790 4792 4793 4796 4798 4799 4804 4810
 4811 4818 5038 5070 5110 5186 5198 5250 5298 5358 5438 5500 5590 5610
 5618 5659 5706 5758 5766 5

seed 9: acc=0.9282  MCC=0.8567

acc 0.9452 ± 0.0101
MCC 0.8906 ± 0.0204


In [22]:
# ===============================
# Treshold + Calibration analysis
# ===============================
from sklearn.metrics import matthews_corrcoef, precision_score, recall_score
from sklearn.calibration import calibration_curve

# 1) choose the threshold on a validation split taken from the TRAINING clusters
X_train_part = X.iloc[train_idx]
y_train_part = y.iloc[train_idx]
groups_train_part = groups.iloc[train_idx]

inner_train_idx, validation_idx = next(
    GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
    .split(X_train_part, y_train_part, groups_train_part)
)

pipeline_for_threshold = clone(pipe_xgb)
pipeline_for_threshold.fit(X_train_part.iloc[inner_train_idx],
                           y_train_part.iloc[inner_train_idx])

y_validation = np.asarray(y_train_part.iloc[validation_idx])
y_validation_prob = pipeline_for_threshold.predict_proba(
    X_train_part.iloc[validation_idx])[:, 1]

thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)
best_threshold = max(
    thresholds,
    key=lambda threshold: matthews_corrcoef(
        y_validation, (y_validation_prob >= threshold).astype(int))
)
print(f"threshold chosen on validation: {best_threshold:.2f}\n")

# 2) sweep the threshold on the test set, with best_threshold already fixed
y_test_array = np.asarray(y_test)
rows = []
for threshold in thresholds:
    y_pred_at_threshold = (y_prob >= threshold).astype(int)
    rows.append({
        "threshold": threshold,
        "precision": precision_score(y_test_array, y_pred_at_threshold, zero_division=0),
        "recall":    recall_score(y_test_array, y_pred_at_threshold),
        "MCC":       matthews_corrcoef(y_test_array, y_pred_at_threshold),
        "note":      "chosen" if threshold == best_threshold
                     else ("default" if threshold == 0.5 else ""),
    })
sweep_table = pd.DataFrame(rows).round(4)
display(sweep_table)

# 3) calibration: mean predicted probability vs observed frequency, 10 bins
observed_frequency, mean_predicted_prob = calibration_curve(
    y_test_array, y_prob, n_bins=10, strategy="quantile")

calibration_table = pd.DataFrame({
    "mean_predicted_probability": mean_predicted_prob,
    "observed_frequency": observed_frequency,
}).round(3)
display(calibration_table)

c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 510  639  678  718  782  786  792  798  858  870  958  978 1018 1026
 1029 1030 1037 1038 1039 1057 1070 1087 1090 1098 1183 1192 1193 1194
 1218 1253 1318 1346 1350 1378 1421 1426 1430 1458 1510 1513 1558 1581
 1586 1602 1701 1718 1758 1838 1894 1938 1958 1980 1981 1986 1998 2038
 2050 2058 2098 2158 2178 2226 2227 2230 2239 2258 2290 2310 2318 2370
 2378 2383 2390 2391 2394 2396 2397 2398 2399 2406 2646 2838 2850 2890
 2918 2930 2958 2970 2991 3018 3020 3023 3026 3030 3035 3036 3038 3043
 3050 3070 3098 3118 3150 3159 3166 3178 3180 3186 3189 3190 3194 3195
 3196 3197 3198 3199 3253 3290 3350 3424 3430 3438 3441 3484 3510 3558
 3582 3587 3592 3593 3598 3599 3706 3824 3826 3858 3891 3994 3996 4098
 4147 4158 4178 4224 4258 4298 4310 4381 4392 4398 4446 4450 4451 4456
 4459 4464 4472 4473 4478 4498 4503 4510 4512 4518 4538 4541 4546 4548
 4550 4551 4552 4553 4554 4

threshold chosen on validation: 0.40



,threshold,precision,recall,MCC,note
0,0.05,0.8656,0.9728,0.8281,
1,0.10,0.8948,0.9637,0.8530,
2,0.15,0.9163,0.9592,0.8726,
3,0.20,0.9280,0.9547,0.8810,
4,0.25,0.9377,0.9547,0.8914,
5,0.30,0.9472,0.9486,0.8958,
6,0.35,0.9543,0.9471,0.9018,
7,0.40,0.9617,0.9471,0.9095,chosen
8,0.45,0.9630,0.9426,0.9065,
9,0.50,0.9626,0.9335,0.8977,default


,mean_predicted_probability,observed_frequency
0,0.000,0.008
1,0.000,0.023
2,0.002,0.030
3,0.008,0.023
4,0.101,0.174
5,0.800,0.773
6,0.996,0.977
7,1.000,0.992
8,1.000,1.000
9,1.000,1.000
